In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Univariate Analysis

## Numeric variables
- Summary stats: count, mean, median, std, min, max, quartiles, IQR, skewness.
- Visuals: histogram, boxplot, density plot.
- Check for multimodality (multiple peaks) and heavy tails.

## Categorical variables
- Frequency counts
- Bar charts and Pareto charts
- Check rare categories (low-frequency levels) and cardinality.

## Date/time variables
- Check for time range gaps, time granularity
- Visuals: time series plot, seasonality checks, Heatmap (for example: day vs hour), Bar chart (month/year)

## Text variables
- Token counts, most common words, length distributions
- Histogram of text length
- Word cloud / bar of top words

In [ ]:
numeric_cols = ['Age', 'Purchase_Amt']
summary = df['numeric_cols'].describe().T
summary['median'] = df[numeric_cols].median()
summary['IQR'] = summary['75%'] - summary['25%']
summary['skewness'] = df[numeric_cols].skew()
summary                        

In [ ]:
for col in numeric_cols:
    plt.figure(figsize=(12,4))
    
    plt.subplot(1,3,1)
    sns.histplot(df[col], kde=True)
    plt.title(f"Histogram of {col}")
    
    plt.subplot(1,3,2)
    sns.boxplot(y=df[col])
    plt.title(f"Boxplot of {col}")
    
    plt.subplot(1,3,3)
    sns.kdeplot(df[col], fill=True)
    plt.title(f"Density Plot of {col}")
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Frequency count for categorical columns
cat_cols = ['Gender', 'City']

for index, col in enumerate(cat_cols):
    print(f"Frequency counts for {col}:\n", df[col].value_counts())
    plt.figure(figsize=(6,4))
    sns.countplot(x=col, data=df, order=df[col].value_counts().index)
    plt.title(f"Bar chart of {col}")
    plt.show()

In [ ]:
date_cols = ['Signup_date', 'Last_purchase_date']

for col in date_cols:
    print(f"\n Date Range for {col}: {df[col].min()} to {df[col].max(0)}")
    print(f"Time span: {(df[col].max() - df[col].min()).days} days")
    
#Month distribution
plt.figure(figsize = (12,4))
df['Signup_Month'] = df['Signup_date'].dt.month_name()
sns.countplot(x='Signup_Month', data=df)
plt.title('Signup Month Distribution')
plt.show()

#Year distribution
plt.figure(figsize = (4,4))
df['Signup_Year'] = df['Signup_date'].dt.year
sns.countplot(x='Signup_Year', data=df)
plt.title('Signup Year Distribution')
plt.show()

In [ ]:
def time_granularity(series):
    diffs = series.sort_values().diff().dropna().dt.days
    return diffs.describe()

In [ ]:
time_granularity(df['Last_purchase_date'])

In [ ]:
df['Last_purchase_date'].sort_values().diff().value_counts()

In [ ]:
df.groupby('Last_purchase_date')[['Purchase_Amt']].sum().plot()
plt.show()

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

df_daily = df.groupby('Last_purchase_date')['Purchase_Amt'].sum()
result = seasonal_decompose(df_daily, model='additive', period=30)
result.plot().show()

In [ ]:
df['day'] = df['Last_purchase_date'].dt.day
df['month'] = df['Last_purchase_date'].dt.month
heatmap_data = df.pivot_table(
    index='day',
    columns='month',
    values='Purchase_Amt',
    aggfunc='mean'
)

# Plot heatmap
plt.figure(figsize=(12,12))
sns.heatmap(heatmap_data, cmap='YlGnBu', annot=True, fmt='.1f')
plt.xlabel('Month')
plt.ylabel('Day of Month')
plt.title('Purchase Heatmap: Day vs Month')
plt.show()

In [ ]:
import re
# Text preprocessing
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

text_df['clean_text'] = clean_text['text'].apply(clean_text)

In [ ]:
text_df['tokens'] = text_df['clean_text'].apply(lambda x:x.split())

In [ ]:
# Histogram of text lengths
plt.figure()
plt.hist(text_df['token_count'])
plt.xlabel('Number of Words per Review')
plt.ylabel('Frequency')
plt.title('Histogram of Review Text Lengths')
plt.show()

In [ ]:
# most common words
all_tokens = [token for tokens in text_df['tokens'] for token in tokens]
word_freq = Counter(all_tokens)

top_words = word_freq.most_common(10)
top_words_df = pd.DataFrame(top_words, columns=['word', 'count'])

print('\nTop 10 Most Common Words')
print(top_words_df)

In [ ]:
# after removing stopwords
from nltk.corpus import stopwords
stopword = set(stopwords.words())
all_tokens = [token for token in all_tokens if token not in stopword]
word_freq = Counter(all_tokens)

top_words = word_freq.most_common(10)
top_words_df = pd.DataFrame(top_words, columns=['word', 'count'])

print('\nTop 10 Most Common Words')
print(top_words_df)

plt.figure()
plt.bar(top_words_df['word'], top_words_df['count'])
plt.xlabel('Words')
plt.ylabel('Frequency')
plt.title('Top 10 Most Common Words')
plt.xticks(rotation=45)
plt.show()

# Bivariate Analysis

## Numeric - Numeric
- Scatter plots,, correlation coefficient
- Regression line to visualize trend

## Numeric - Categorical
- Boxplots or violin plots grouped by category
- Compute group means/medians
- ANOVA for significant differences

## Categorical - Categorical
- Contingency tables, stacked bar charts
- Chi-square test of independence

## Time series relationship
- Lag plots, autocorrelation (ACF/PACF)
- Rolling averages

In [ ]:
corr = weather_df.corr()

plt.figure(figsize=(5,4))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

# Pairwise scatter plots + regression line
sns.pairplot(weather_df, kind='reg', diag_kind='kde', corner=True, plot_kws={'line_kws':{'color':'red'}})
plt.subtitle('Scatter and Regression Lines Between Numeric Variables', y=1.02)
plt.show()

In [ ]:
cat_cols = ['Gender', 'City']

for cat in cat_cols:
    for num in numeric_cols:
        plt.figure(figsize=(7,2))
        sns.boxplot(x=cat, y=num, data=df)
        plt.title(f'{num} by {cat}')
        plt.show()
        
        print(f'\n Group Mean and Median of {num} by {cat}:')
        print(df.groupby(cat)[num].agg(['mean','median','count']))

In [ ]:
from scipy import stats
anova_res = stats.f_oneway(*[group['Feedback_Score'].values for name, group in df.groupby('City')])
print('ANOVA Test for Feedback_Score across Cities:')
print('F-statistic:', round(anova_res.statistic, 3), ", p-value:", round(anova_res.pvalue, 3))

In [ ]:
contigency = pd.crosstab(df['Gender'], df['City'])
print('\n Contingency Table (Gender vs City):\n', contingency)

# Stacked Bar Chart
contingency.plot(kind='bar', stacked=True, figsize=(8,5))
plt.title('Stacked Bar Chart: Gender vs City')
plt.ylabel('Count')
plt.show()

# Chi-square Test of Independence
chi2, p, dof, expected = stats.chi2_contingency(contingency)
print('\nChi-Square Test Results:')
print(f'Chi2 = {chi2:.3f}, p-value = {p:.3f}, Degrees of Freedom = {dof}')

if p < 0.05:
    print('Significant relationship (reject null hypothesis)')
else:
    print('No significant relationship (fail to reject null hypothesis)')

In [ ]:
from pandas.plotting import lag_plot

plt.figure()
lag_plot(sales_df['sales'], lag=1)
plt.title('Lag Plot (Lag = 1 day)')
plt.show()

plt.figure()
lag_plot(sales_df['sales'], lag=7)
plt.title('Lag Plot (Lag = 7 days)')
plt.show()

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

plt.figure()
plot_acf(sales_df['sales'], lags=30)
plt.title('Autocorrelation (ACF) of Daily Sales')
plt.show()

In [ ]:
from statsmodels.graphics.tsaplots import plot_pacf

plt.figure()
plot_pacf(sales_df['sales'], lags=40, method='ywm')
plt.title('PACF - Daily Sales')
plt.show()

In [ ]:
sales_df['rolling_7'] = sales_df['sales'].rolling(window=7).mean()
sales_df['rolling_14'] = sales_df['sales'].rolling(window=14).mean()

plt.figure()
plt.plot(sales_df.index, sales_df['sales'], label='Daily Sales')
plt.plot(sales_df.index, sales_df['rolling_7'], label='7-day Rolling Avg')
plt.plot(sales_df.index, sales_df['rolling_14'], label='14-day Rolling Avg')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.title('Rolling Average Analysis')
plt.legend()
plt.show()

# Multivariate Analysis

- Heatmap of correlations (numeric)
- Interaction plots (for interactions between two features affecting target)
- Dimensionality reduction: PCA/UMAP/t-SNE to inspect clusters or structure.
- Clustering to find natural groups.
- Regression / Predictive Relationships.

In [ ]:
# Example: Age + Gender -> Purchase Amount
plt.figure(figsize=(8,5))
sns.boxplot(
    data=df,
    x='Gender',
    y='Purchase_Amt',
    hue='Age',
    dodge=True
)
plt.title('Interaction Plot: Gender & Age vs Purchase Amount')

In [ ]:
# Example: City + Feedback -> Purchase Amount
plt.figure(figsize=(8,5))
sns.boxplot(
    data=df,
    x='City',
    y='Purchase_Amt',
    hue='Feedback_Score',
    dodge=True
)
plt.title('Interaction Plot: City & Feedback vs Purchase Amount')

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
X = df['numeric_cols']

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

df['PCA1'] = X_pca[:,0]
df['PCA2'] = X_pca[:,1]

print('Explained Variance Ratio:', pca.explained_variance_ratio_)

In [ ]:
plt.figure(figsize=(7,5))
sns.scatterplot(
    x='PCA1',
    y='PCA2',
    data=df,
    s=100
)
plt.title('PCA Projection (2D)')
plt.show()

In [ ]:
# KMeans clustering
kmeans = KMeans(n_clusters=2, random_state=42)
df['Cluster'] = kmeans.fit_predict(X_pca)
plt.figure(figsize=(7,5))
sns.scatterplot(
    x='PCA1',
    y='PCA2',
    hue='Cluster',
    data=df,
    palette='Set1',
    s=100
)
plt.title('Customer Segments using KMeans')
plt.show()

In [ ]:
df.groupby('Cluster')[numeric_cols].mean()